In [ ]:
import pickle
import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.decomposition import PCA
import joblib
import os

# ============================================================
# STEP 1: LOAD DATA
# ============================================================
print("Loading dataset...")

with open('gen-two/fd_data_gen_two_s11.pickle', 'rb') as f:
    fd_data = pickle.load(f)

with open('gen-two/metadata_gen_two.pickle', 'rb') as f:
    metadata = pickle.load(f)

def has_tumor(m):
    r = m['tum_rad']
    try:
        return float(r) > 0
    except (TypeError, ValueError):
        return False

labels = np.array([1 if has_tumor(m) else 0 for m in metadata])

# ============================================================
# STEP 2: BETTER FEATURE EXTRACTION (was the main problem)
# ============================================================
# Instead of just mean → extract multiple statistics per scan
# This gives the model much richer information

mag = np.abs(fd_data)   # shape: (1301, 1001, 72)

print("Extracting rich features...")

# --- Statistical features across frequency axis ---
feat_mean  = mag.mean(axis=1)          # mean per antenna     → (1301, 72)
feat_std   = mag.std(axis=1)           # std per antenna      → (1301, 72)
feat_max   = mag.max(axis=1)           # max per antenna      → (1301, 72)
feat_min   = mag.min(axis=1)           # min per antenna      → (1301, 72)
feat_range = feat_max - feat_min       # range per antenna    → (1301, 72)

# --- Frequency band features (split into 3 bands) ---
# Low: 0-33%, Mid: 33-66%, High: 66-100% of frequency axis
n_freq = mag.shape[1]
b1 = mag[:, :n_freq//3,    :].mean(axis=1)   # low band   → (1301, 72)
b2 = mag[:, n_freq//3:2*n_freq//3, :].mean(axis=1)  # mid → (1301, 72)
b3 = mag[:, 2*n_freq//3:,  :].mean(axis=1)   # high band  → (1301, 72)

# --- Also add mean across antennas (original feature) ---
mean_across_ant = mag.mean(axis=2)            # → (1301, 1001)

# --- Concatenate all statistical features ---
features_rich = np.hstack([
    feat_mean,    # 72
    feat_std,     # 72
    feat_max,     # 72
    feat_min,     # 72
    feat_range,   # 72
    b1,           # 72
    b2,           # 72
    b3,           # 72
])
# Total: 72 × 8 = 576 features

print(f"Rich feature shape: {features_rich.shape}")  # (1301, 576)
print(f"Healthy: {np.sum(labels==0)} | Tumor: {np.sum(labels==1)}")

# ============================================================
# STEP 3: PCA — reduce 576 → 100 components (keeps 95% info)
# ============================================================
X_train_r, X_test_r, y_train, y_test = train_test_split(
    features_rich, labels, test_size=0.2, random_state=42, stratify=labels
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_r)
X_test_sc  = scaler.transform(X_test_r)

pca = PCA(n_components=100, random_state=42)
X_train_pca = pca.fit_transform(X_train_sc)
X_test_pca  = pca.transform(X_test_sc)

print(f"After PCA: {X_train_pca.shape}")
print(f"Variance explained: {pca.explained_variance_ratio_.sum()*100:.1f}%")

# ============================================================
# STEP 4: TRAIN MODELS ON RICH FEATURES
# ============================================================
models = {
    "SVM (RBF)"        : SVC(kernel='rbf', C=50, gamma='scale',
                              class_weight='balanced', probability=True),
    "Random Forest"    : RandomForestClassifier(n_estimators=300,
                              max_depth=15, class_weight='balanced',
                              random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=300,
                              learning_rate=0.05, max_depth=4, random_state=42),
    "KNN"              : KNeighborsClassifier(n_neighbors=5,
                              weights='distance'),
}

print("\n" + "="*55)
print("  INDIVIDUAL MODEL RESULTS")
print("="*55)

trained_models = {}
for name, model in models.items():
    model.fit(X_train_pca, y_train)
    y_pred = model.predict(X_test_pca)
    acc = (y_pred == y_test).mean() * 100
    print(f"\n📌 {name}  —  Accuracy: {acc:.1f}%")
    print(classification_report(y_test, y_pred,
          target_names=['Healthy','Tumor'], digits=3))
    trained_models[name] = model

# ============================================================
# STEP 5: ENSEMBLE
# ============================================================
ensemble = VotingClassifier(
    estimators=[
        ('svm', trained_models["SVM (RBF)"]),
        ('rf',  trained_models["Random Forest"]),
        ('gb',  trained_models["Gradient Boosting"]),
        ('knn', trained_models["KNN"]),
    ],
    voting='soft'
)
ensemble.fit(X_train_pca, y_train)
y_ens = ensemble.predict(X_test_pca)
ens_acc = (y_ens == y_test).mean() * 100

print("="*55)
print(f"\n✅ Ensemble Accuracy: {ens_acc:.1f}%")
print(classification_report(y_test, y_ens,
      target_names=['Healthy','Tumor'], digits=3))

# ============================================================
# STEP 6: SAVE EVERYTHING (including PCA)
# ============================================================
os.makedirs('saved_models', exist_ok=True)

joblib.dump(scaler,                              'saved_models/scaler.pkl')
joblib.dump(pca,                                 'saved_models/pca.pkl')
joblib.dump(trained_models["SVM (RBF)"],         'saved_models/model_svm.pkl')
joblib.dump(trained_models["Random Forest"],     'saved_models/model_rf.pkl')
joblib.dump(trained_models["Gradient Boosting"], 'saved_models/model_gb.pkl')
joblib.dump(trained_models["KNN"],               'saved_models/model_knn.pkl')
joblib.dump(ensemble,                            'saved_models/model_ensemble.pkl')

# Save feature extraction info for use in predict function
feature_info = {
    'n_freq'     : n_freq,
    'target_len' : mean_across_ant.shape[1],   # 1001
}
joblib.dump(feature_info, 'saved_models/feature_info.pkl')

print("\n✅ All models + PCA + scaler saved!")

# ============================================================
# STEP 7: REGENERATE TEST FILES WITH CORRECT RICH FEATURES
# ============================================================
tumor_indices   = np.where(labels == 1)[0]
healthy_indices = np.where(labels == 0)[0]

def extract_rich_features_single(scan_3d):
    """
    scan_3d: shape (1001, 72) — one patient scan
    Returns: 1D feature vector of length 576
    """
    m = np.abs(scan_3d)                         # (1001, 72)
    n = m.shape[0]

    f_mean  = m.mean(axis=0)                    # (72,)
    f_std   = m.std(axis=0)
    f_max   = m.max(axis=0)
    f_min   = m.min(axis=0)
    f_range = f_max - f_min
    f_b1    = m[:n//3,    :].mean(axis=0)       # low band
    f_b2    = m[n//3:2*n//3, :].mean(axis=0)   # mid band
    f_b3    = m[2*n//3:,  :].mean(axis=0)       # high band

    return np.concatenate([f_mean, f_std, f_max, f_min,
                           f_range, f_b1, f_b2, f_b3])  # (576,)

# Save tumor test file
t_idx    = tumor_indices[5]   # pick index 5 (larger tumor, easier to detect)
t_feat   = extract_rich_features_single(fd_data[t_idx])
t_meta   = metadata[t_idx]
pd.DataFrame(t_feat.reshape(1, -1)).to_csv('patient_tumor.csv',
                                            index=False, header=False)
print(f"\n✅ patient_tumor.csv  — tum_rad: {t_meta['tum_rad']} cm")

# Save healthy test file
h_idx    = healthy_indices[5]
h_feat   = extract_rich_features_single(fd_data[h_idx])
pd.DataFrame(h_feat.reshape(1, -1)).to_csv('patient_healthy.csv',
                                            index=False, header=False)
print(f"✅ patient_healthy.csv — no tumor")

# ============================================================
# STEP 8: UPDATED PREDICT FUNCTION
# ============================================================
def predict_patient_file(filepath):
    scaler_   = joblib.load('saved_models/scaler.pkl')
    pca_      = joblib.load('saved_models/pca.pkl')
    feat_info = joblib.load('saved_models/feature_info.pkl')

    if filepath.endswith('.csv'):
        df = pd.read_csv(filepath, header=None)
    elif filepath.endswith(('.xlsx', '.xls')):
        df = pd.read_excel(filepath, header=None)
    else:
        print("❌ Unsupported format"); return

    data = df.values.astype(float).flatten()
    print(f"\n📂 File: {filepath}  |  Features: {len(data)}")

    # Resize if needed
    target = 576   # our rich feature vector length
    if len(data) != target:
        from scipy.interpolate import interp1d
        data = interp1d(np.linspace(0,1,len(data)),
                        data)(np.linspace(0,1,target))
        print(f"   ⚠️  Resampled to {target} features")

    # Scale + PCA
    data_sc  = scaler_.transform(data.reshape(1, -1))
    data_pca = pca_.transform(data_sc)

    # --- Each model votes ---
    print("\n" + "="*48)
    print("  🔬 DIAGNOSIS REPORT")
    print("="*48)

    model_map = {
        "SVM (RBF)"        : "model_svm",
        "Random Forest"    : "model_rf",
        "Gradient Boosting": "model_gb",
        "KNN"              : "model_knn",
    }

    votes, confidences = [], []
    for name, key in model_map.items():
        mdl  = joblib.load(f'saved_models/{key}.pkl')
        pred = mdl.predict(data_pca)[0]
        prob = mdl.predict_proba(data_pca)[0]
        conf = prob[pred] * 100
        tag  = "🔴 TUMOR" if pred == 1 else "🟢 HEALTHY"
        print(f"  {name:<24} → {tag}  ({conf:.1f}%)")
        votes.append(pred)
        confidences.append(conf)

    ens  = joblib.load('saved_models/model_ensemble.pkl')
    ep   = ens.predict(data_pca)[0]
    eprob= ens.predict_proba(data_pca)[0]
    ec   = eprob[ep] * 100

    tv = sum(votes); hv = 4 - tv

    print("\n" + "-"*48)
    print(f"  Votes  →  🔴 Tumor: {tv}/4   🟢 Healthy: {hv}/4")
    print("-"*48)
    verdict = "🔴 TUMOR DETECTED" if ep == 1 else "🟢 HEALTHY — No Tumor"
    print(f"\n  ✅ FINAL VERDICT : {verdict}")
    print(f"     Confidence    : {ec:.1f}%")

    if ec < 60:
        print("  ⚠️  Low confidence — recommend clinical review")
    if tv >= 3 and ep == 1:
        print("  ⚠️  Strong model agreement — high suspicion of tumor")
    elif tv == 2:
        print("  ⚠️  Mixed results — further examination recommended")

    print("="*48)
    return {"verdict": "Tumor" if ep==1 else "Healthy",
            "confidence": round(ec,2),
            "votes": {"tumor": tv, "healthy": hv}}

# ============================================================
# TEST
# ============================================================
print("\n--- Testing TUMOR patient ---")
predict_patient_file('patient_tumor.csv')

print("\n--- Testing HEALTHY patient ---")
predict_patient_file('patient_healthy.csv')

: 

In [ ]:
### Generate Test File

In [ ]:
import pickle
import numpy as np
import pandas as pd

# ============================================================
# LOAD THE DATASET
# ============================================================
with open('gen-two/fd_data_gen_two_s11.pickle', 'rb') as f:
    fd_data = pickle.load(f)

with open('gen-two/metadata_gen_two.pickle', 'rb') as f:
    metadata = pickle.load(f)

def has_tumor(m):
    r = m['tum_rad']
    try:
        return float(r) > 0
    except (TypeError, ValueError):
        return False

labels = np.array([1 if has_tumor(m) else 0 for m in metadata])

# Mean over antenna axis → (1301, 1001)
features = np.abs(fd_data).mean(axis=2)

# ============================================================
# FIND REAL SAMPLES FROM THE DATASET
# ============================================================
tumor_indices   = np.where(labels == 1)[0]
healthy_indices = np.where(labels == 0)[0]

print(f"Available tumor samples  : {len(tumor_indices)}")
print(f"Available healthy samples: {len(healthy_indices)}")

# ============================================================
# EXPORT SINGLE PATIENT FILES (for testing predict_patient_file)
# ============================================================

# --- Pick one TUMOR patient ---
tumor_idx   = tumor_indices[0]       # change index to pick different patients
tumor_sample = features[tumor_idx]   # shape: (1001,)
tumor_meta   = metadata[tumor_idx]

df_tumor = pd.DataFrame(tumor_sample.reshape(1, -1))
df_tumor.to_csv('patient_tumor.csv', index=False, header=False)
print(f"\n✅ Saved: patient_tumor.csv")
print(f"   tum_rad : {tumor_meta['tum_rad']} cm")
print(f"   birads  : {tumor_meta['birads']}")
print(f"   phant_id: {tumor_meta['phant_id']}")

# --- Pick one HEALTHY patient ---
healthy_idx    = healthy_indices[0]
healthy_sample = features[healthy_idx]
healthy_meta   = metadata[healthy_idx]

df_healthy = pd.DataFrame(healthy_sample.reshape(1, -1))
df_healthy.to_csv('patient_healthy.csv', index=False, header=False)
print(f"\n✅ Saved: patient_healthy.csv")
print(f"   tum_rad : {healthy_meta['tum_rad']} (no tumor)")
print(f"   birads  : {healthy_meta['birads']}")
print(f"   phant_id: {healthy_meta['phant_id']}")

# ============================================================
# EXPORT BATCH TEST FILE (multiple patients at once)
# ============================================================

# Pick 10 tumor + 10 healthy samples
n = 10
batch_tumor   = features[tumor_indices[:n]]
batch_healthy = features[healthy_indices[:n]]

batch_X = np.vstack([batch_tumor, batch_healthy])
batch_y = np.array([1]*n + [0]*n)

df_batch = pd.DataFrame(batch_X)
df_batch['true_label'] = ['Tumor']*n + ['Healthy']*n
df_batch.to_csv('batch_test_patients.csv', index=False)
print(f"\n✅ Saved: batch_test_patients.csv")
print(f"   {n} tumor + {n} healthy = {n*2} total patients")

# ============================================================
# ALSO EXPORT AS EXCEL
# ============================================================
df_tumor.to_excel('patient_tumor.xlsx',   index=False, header=False)
df_healthy.to_excel('patient_healthy.xlsx', index=False, header=False)
print(f"\n✅ Also saved .xlsx versions")

# ============================================================
# TEST IMMEDIATELY
# ============================================================
print("\n" + "="*50)
print("TESTING TUMOR PATIENT FILE:")
predict_patient_file('patient_tumor.csv')

print("\n" + "="*50)
print("TESTING HEALTHY PATIENT FILE:")
predict_patient_file('patient_healthy.csv')